## Example 1: JWT and OpenID Connect support

In [ ]:
# Example 1: JWT and OpenID Connect Support
import oauthlib.oauth2
from oauthlib.oauth2 import is_secure_transport
import oauthlib.openid
import json
import base64
import os
import time

print("Example 1: JWT tokens and OpenID Connect functionality")

try:
    # Test secure transport validation
    print("Testing secure transport validation:")
    test_uris = [
        ('https://secure.example.com', True),
        ('http://insecure.example.com', False),
        # oauthlib has no localhost exception: is_secure_transport() is True only for https:// unless the
        # process opts out with OAUTHLIB_INSECURE_TRANSPORT.
        ('http://localhost:8000', False),
        ('http://127.0.0.1:3000', False)
    ]
    
    saved_insecure_transport = os.environ.pop('OAUTHLIB_INSECURE_TRANSPORT', None)
    try:
        for uri, expected in test_uris:
            is_secure = is_secure_transport(uri)
            print(f"  {uri}: {is_secure} (expected: {expected})")
            print(f"    Result: {'✓' if is_secure == expected else '✗'}")
            assert is_secure == expected, (uri, is_secure)
        # The documented opt-out for local development: plain http is then accepted.
        os.environ['OAUTHLIB_INSECURE_TRANSPORT'] = '1'
        assert is_secure_transport('http://localhost:8000'), 'OAUTHLIB_INSECURE_TRANSPORT=1 did not allow plain http'
        print("  OAUTHLIB_INSECURE_TRANSPORT=1 allows http://localhost:8000")
    finally:
        os.environ.pop('OAUTHLIB_INSECURE_TRANSPORT', None)
        if saved_insecure_transport is not None:
            os.environ['OAUTHLIB_INSECURE_TRANSPORT'] = saved_insecure_transport
    
    # Test OpenID Connect client
    print(f"\nTesting OpenID Connect client:")
    try:
        # oauthlib.openid ships only the PROVIDER side (Server, RequestValidator, UserInfoEndpoint); the
        # OIDC authorization-code request is built by the OAuth2 WebApplicationClient, with the OIDC
        # `nonce` passed as an extra request parameter.
        oidc_client = oauthlib.oauth2.WebApplicationClient('test-client-id')
        print(f"OpenID Connect client created: {oidc_client.__class__.__name__}")
        
        # Test OpenID Connect authorization URL with nonce
        auth_url = oidc_client.prepare_request_uri(
            'https://auth.example.com/oauth/authorize',
            redirect_uri='https://client.example.com/callback',
            scope=['openid', 'profile', 'email'],
            nonce='test-nonce-12345',
            state='test-state-67890'
        )
        
        print(f"OIDC Authorization URL: {auth_url[:80]}...")
        print(f"Contains nonce: {'nonce=' in auth_url}")
        print(f"Contains openid scope: {'openid' in auth_url}")
        assert 'nonce=test-nonce-12345' in auth_url, auth_url
        assert 'scope=openid+profile+email' in auth_url, auth_url
        assert 'response_type=code' in auth_url, auth_url
        
    except Exception as e:
        print(f"OpenID Connect testing: {e}")
        raise
    
    # Simulate JWT token structure (for educational purposes)
    print(f"\nSimulating JWT token structure:")
    
    # JWT Header
    jwt_header = {
        'typ': 'JWT',
        'alg': 'RS256',
        'kid': 'key-id-12345'
    }
    
    # JWT Payload (ID Token)
    current_time = int(time.time())
    jwt_payload = {
        'iss': 'https://auth.example.com',
        'sub': 'user-12345',
        'aud': 'test-client-id',
        'exp': current_time + 3600,  # 1 hour
        'iat': current_time,
        'nonce': 'test-nonce-12345',
        'name': 'Test User',
        'email': 'test@example.com',
        'email_verified': True
    }
    
    print("JWT ID Token structure:")
    print(f"  Header: {json.dumps(jwt_header, indent=None)}")
    print(f"  Payload claims: {len(jwt_payload)}")
    for claim, value in jwt_payload.items():
        print(f"    {claim}: {value}")
    
    # Base64 encode components (without actual signing)
    header_b64 = base64.urlsafe_b64encode(json.dumps(jwt_header).encode()).decode().rstrip('=')
    payload_b64 = base64.urlsafe_b64encode(json.dumps(jwt_payload).encode()).decode().rstrip('=')
    
    print(f"\nJWT components (Base64 encoded):")
    print(f"  Header: {header_b64[:30]}...")
    print(f"  Payload: {payload_b64[:30]}...")
    print(f"  Signature: [would be cryptographically signed]")
    
except Exception as e:
    raise AssertionError(
        f"oauthlib example 1 failed: {type(e).__name__}: {e}"
    ) from e

print("JWT and OpenID Connect testing completed successfully")

## Example 2: security features and error handling

In [ ]:
# Example 2: Security Features and Error Handling
import oauthlib.oauth1
import oauthlib.oauth2
import oauthlib.common
from oauthlib.oauth2.rfc6749 import errors

print("Example 2: Security features, error handling, and utility functions")

try:
    # Test OAuth security utilities
    print("Testing OAuth security utilities:")
    
    # Generate secure random strings
    random_client_id = oauthlib.common.generate_client_id()
    random_token = oauthlib.common.generate_token()
    
    print(f"Generated client ID: {random_client_id}")
    print(f"Generated token: {random_token[:20]}...")
    print(f"Client ID length: {len(random_client_id)}")
    print(f"Token length: {len(random_token)}")
    assert len(random_client_id) == 30 and len(random_token) == 30, (random_client_id, random_token)
    
    # Test parameter encoding/decoding
    test_params = [
        ('client_id', 'test-client-123'),
        ('redirect_uri', 'https://app.example.com/callback'),
        ('scope', 'read write admin'),
        ('state', 'random state with spaces')
    ]
    
    print(f"\nTesting parameter encoding:")
    encoded_params = []
    for key, value in test_params:
        encoded = oauthlib.common.quote(str(value), safe='')
        decoded = oauthlib.common.unquote(encoded)
        encoded_params.append((key, encoded))
        
        print(f"  {key}:")
        print(f"    Original: {value}")
        print(f"    Encoded: {encoded}")
        print(f"    Decoded: {decoded}")
        print(f"    Round-trip: {'✓' if value == decoded else '✗'}")
        assert value == decoded, (key, value, encoded, decoded)
    
    # Test OAuth 2.0 error handling
    print(f"\nTesting OAuth 2.0 error types:")
    
    oauth2_errors = [
        ('InvalidClientError', errors.InvalidClientError),
        ('InvalidGrantError', errors.InvalidGrantError),
        ('InvalidRequestError', errors.InvalidRequestError),
        ('InvalidScopeError', errors.InvalidScopeError),
        ('UnsupportedGrantTypeError', errors.UnsupportedGrantTypeError),
        ('AccessDeniedError', errors.AccessDeniedError)
    ]
    
    for error_name, error_class in oauth2_errors:
        try:
            error_instance = error_class(description=f"Test {error_name}")
            print(f"  {error_name}:")
            print(f"    Error code: {getattr(error_instance, 'error', 'N/A')}")
            print(f"    Status code: {getattr(error_instance, 'status_code', 'N/A')}")
            print(f"    Description: {getattr(error_instance, 'description', 'N/A')}")
            expected_status = 401 if error_class is errors.InvalidClientError else 400
            assert error_instance.status_code == expected_status, (error_name, error_instance.status_code)
            
        except Exception as e:
            print(f"  {error_name}: Error creating - {e}")
            raise
    
    # Test request validation utilities
    print(f"\nTesting request validation:")
    
    # Test case-insensitive dict
    headers = oauthlib.common.CaseInsensitiveDict({
        'Authorization': 'Bearer token123',
        'Content-Type': 'application/x-www-form-urlencoded',
        'User-Agent': 'OAuth-Test/1.0'
    })
    
    print(f"Case-insensitive header access:")
    test_keys = ['authorization', 'CONTENT-TYPE', 'user-agent', 'Accept']
    for key in test_keys:
        value = headers.get(key, 'Not found')
        print(f"  headers['{key}']: {value}")
        assert (value == 'Not found') == (key == 'Accept'), (key, value)
    
    # Test parameter extraction
    query_string = 'client_id=test&response_type=code&scope=read+write&state=xyz123'
    extracted_params = oauthlib.common.extract_params(query_string)
    
    print(f"\nParameter extraction from query string:")
    print(f"Query: {query_string}")
    print(f"Extracted parameters: {len(extracted_params)}")
    assert extracted_params == [('client_id', 'test'), ('response_type', 'code'), ('scope', 'read write'), ('state', 'xyz123')], (
        extracted_params
    )
    for param_list in extracted_params:
        print(f"  {param_list[0]}: {param_list[1]}")
    
except Exception as e:
    raise AssertionError(
        f"oauthlib example 2 failed: {type(e).__name__}: {e}"
    ) from e

print("Security features and error handling testing completed successfully")